<!-- beginner-banner-v1 -->

> 🧭 **비개발자 수강생 안내** — 이 노트북에서 새로 배우는 것: **BM25+벡터 하이브리드** + Re-rank 로 검색 정밀도 끌어올리기.
>
> - 📖 강의 페이지: [day3/18-advanced-rag-retrieval](https://siapapa.github.io/day3/18-advanced-rag-retrieval/)
> - 🆕 처음이라면 → [비개발자 학습 가이드](https://siapapa.github.io/beginners-guide/)
> - 🔤 모르는 단어 → [용어 사전](https://siapapa.github.io/appendix/glossary/)
> - 🛠️ 환경/접속 막힘 → [사전 준비](https://siapapa.github.io/setup/) · [트러블슈팅](https://siapapa.github.io/appendix/troubleshooting/)
>
> **셀은 위에서 아래로 차례대로 실행**하세요. 시연용 코드(`구경만 하세요` 표시)는 지금 이해 못 해도 100% 정상입니다.

---



# 15. Advanced RAG — 검색 고도화 (Hybrid · Re-rank · Parent-Child)
> Day 3 · 18H · 소요 약 50분

## 학습 목표

- **BM25** (키워드 매칭) 와 **벡터** (의미 유사도) 검색의 차이를 실감하고 `EnsembleRetriever` 로 **하이브리드**를 구성한다.
- **CrossEncoder Re-rank** 로 1 차 후보를 정밀하게 재정렬한다 — 2 단계 검색 (후보 생성 → 재정렬) 패턴.
- **Parent-Child Chunking** (`ParentDocumentRetriever`) — 작은 청크로 정밀 검색, 큰 청크로 문맥 전달.
- BM25 / 벡터 비중 (`weights`) 실험으로 도메인별 최적 비율을 찾는다.

> **설치 시간 경고.** `sentence-transformers` 는 Colab 에서 **2~3 분** 걸립니다 (PyTorch 포함). 첫 실행 시 CrossEncoder 모델 (~80MB) 다운로드도 함께 일어납니다. 수업 흐름을 끊지 않도록 아래 `%pip install` 셀을 **강의 시작 전 미리 실행** 해두세요.

> **DB 미사용.** `./retrieval_chroma` 경로에 새로 적재합니다 — 04/05/10/13/14 와 경로 분리.

In [ ]:
%pip install -q langchain langchain-openai langchain-community langchain-core langchain-chroma chromadb rank_bm25 sentence-transformers

In [ ]:
# Colab/로컬 환경에서 필요한 환경변수를 안전하게 로딩합니다 (다른 노트북과 동일 패턴).
import os

def _load_secret(key: str, required: bool = True) -> None:
    """Colab Secrets → getpass 입력 순으로 시도해 환경변수에 적재."""
    if os.environ.get(key):
        return
    value = None
    try:
        from google.colab import userdata  # type: ignore
        value = userdata.get(key)
    except Exception:
        value = None
    if not value:
        try:
            from getpass import getpass
            value = getpass(f"Enter {key}: ")
        except Exception:
            value = None
    if value:
        os.environ[key] = value
    elif required:
        raise RuntimeError(f"{key} is not set. Register it in Colab Secrets or via env var.")

# Cohere 키는 선택 — 있으면 Cohere Re-rank 사용 가능, 없으면 sentence-transformers CrossEncoder 폴백.
_load_secret("OPENAI_API_KEY", required=True)
print("Environment ready.")

## 1. 문서 코퍼스 + 벡터스토어

병원 안내 문서 10 개를 `./retrieval_chroma` 에 적재합니다. 고유명사(의사 이름), 의미적 질문, 혼합 질문을 모두 커버할 수 있도록 문장을 구성했습니다.

In [ ]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.documents import Document

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

hospital_documents = [
    Document(page_content="심장내과 김철수 전문의는 관상동맥 중재술 1,200례 경험을 보유하고 있으며 매주 수요일 야간 당직을 섭니다.", metadata={"dept": "심장내과"}),
    Document(page_content="호흡기내과 이영희 전문의는 만성 기관지염·천식(ICD-J45) 환자를 주로 진료합니다.", metadata={"dept": "호흡기내과"}),
    Document(page_content="소화기내과 신민아 전문의는 위내시경·대장내시경 전문이며 예약은 2주 전 필수입니다.", metadata={"dept": "소화기내과"}),
    Document(page_content="정형외과 윤성호 전문의는 척추 디스크 수술 전문이며, 한지은 전문의는 관절(무릎·어깨) 전문입니다.", metadata={"dept": "정형외과"}),
    Document(page_content="소아과에는 최동현, 강미래, 문서영 전문의가 있으며 예방접종과 성장 상담을 함께 운영합니다.", metadata={"dept": "소아과"}),
    Document(page_content="응급실은 24시간 운영되며 야간에는 내과 1명·외과 1명이 상주합니다. 심장 응급환자는 즉시 심장내과 당직의가 호출됩니다.", metadata={"type": "emergency"}),
    Document(page_content="외래 진료 시간은 평일 09:00-18:00, 토요일 09:00-13:00 입니다. 점심시간 12:30-13:30 에는 접수가 중단됩니다.", metadata={"type": "schedule"}),
    Document(page_content="입원 병실 요금: 1인실 250,000원/일, 2인실 150,000원/일, 4인실 80,000원/일. 식대 1식 8,000원 별도.", metadata={"type": "admission"}),
    Document(page_content="주차 요금은 외래 3시간 무료, 이후 30분당 1,000원. 입원 보호자는 1일 5,000원 정액입니다.", metadata={"type": "parking"}),
    Document(page_content="외과 박민수(일반외과), 정수진(흉부외과), 권혁준(혈관외과) 전문의가 근무하며 복강경 수술 비중이 60% 이상입니다.", metadata={"dept": "외과"}),
]

vectorstore = Chroma.from_documents(
    hospital_documents,
    embeddings,
    collection_name="retrieval_advanced",
    persist_directory="./retrieval_chroma",
)
print(f"Vectorstore ready: {len(hospital_documents)} docs")

## 2. BM25 vs 벡터 vs 하이브리드

- **BM25** = 단어 빈도 기반. "김철수 의사" 같이 **고유명사**가 들어간 질문에 강함.
- **벡터** = 임베딩 의미 유사도. "호흡기 관련 전문의" 같이 **의미적** 질문에 강함.
- **하이브리드** = 두 점수를 가중 합산. 대부분 질문에서 안정적.

`EnsembleRetriever(weights=[0.4, 0.6])` 는 BM25 40% + 벡터 60% 비중을 의미합니다.

In [ ]:
# BM25 = 키워드 기반 전통적 검색. 단어 빈도 + 역문서빈도(IDF) 로 점수 계산.
# Vector = 임베딩 의미 유사도 검색. 같은 의미 다른 단어를 잘 잡음.
# Ensemble = 두 점수를 weights 비율로 가중 합산. 둘의 약점을 서로 보완.
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

# BM25Retriever.from_documents — 문서 리스트를 받아 인메모리 BM25 인덱스 구축.
bm25_retriever = BM25Retriever.from_documents(hospital_documents)
bm25_retriever.k = 3   # Top-K 설정 (속성 직접 할당 — 다른 retriever 와 인터페이스가 살짝 다름)

# Vector retriever 는 위에서 만든 vectorstore 에서 .as_retriever() 로 추출.
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 두 retriever 를 합쳐 가중 합산 — 점수 정규화·합산은 EnsembleRetriever 가 내부에서 처리.
# weights=[0.4, 0.6] → BM25 40%, 벡터 60%. 도메인에 맞게 0.0~1.0 사이를 실험으로 찾는다.
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.4, 0.6],
)
print("Retrievers ready: bm25 / vector / ensemble (0.4 + 0.6)")

### 비교 테스트 — 3 가지 질문 유형

| 질문 유형 | 예시 |
|---|---|
| 고유명사형 (BM25 우세 기대) | "심장내과 김철수 의사" |
| 의미형 (벡터 우세 기대) | "관절 관련 전문의" |
| 혼합형 (하이브리드 우세 기대) | "야간에 심장 응급환자는 어떻게?" |

In [ ]:
test_questions = [
    ("고유명사", "심장내과 김철수 의사"),
    ("의미", "관절 관련 전문의"),
    ("혼합", "야간에 심장 응급환자는 어떻게?"),
]

for label, q in test_questions:
    print(f"\n{'='*60}\n❓ [{label}] {q}\n{'='*60}")

    bm25 = bm25_retriever.invoke(q)
    print(f"🔵 BM25 ({len(bm25)}개):")
    for doc in bm25:
        print(f"  - {doc.page_content[:60]}...")

    vec = vector_retriever.invoke(q)
    print(f"\n🟢 Vector ({len(vec)}개):")
    for doc in vec:
        print(f"  - {doc.page_content[:60]}...")

    ens = ensemble_retriever.invoke(q)
    print(f"\n🟡 Hybrid ({len(ens)}개):")
    for doc in ens:
        print(f"  - {doc.page_content[:60]}...")

### 한국어 토크나이저 주의

`BM25Retriever.from_documents(...)` 는 기본 공백 토크나이저를 씁니다. 한국어는 조사 (`의사가` / `의사를`) 가 다른 토큰으로 계산돼 점수가 왜곡됩니다. 본인 프로젝트에서는 형태소 분석기 (예: `kiwipiepy`) 를 `preprocess_func` 로 주입하세요.

```python
try:
    from kiwipiepy import Kiwi
    _kiwi = Kiwi()
    def ko_tokenize(text: str) -> list[str]:
        return [t.form for t in _kiwi.tokenize(text) if t.tag.startswith(("N", "V"))]
    bm25_retriever = BM25Retriever.from_documents(docs, preprocess_func=ko_tokenize)
except ImportError:
    bm25_retriever = BM25Retriever.from_documents(docs)
```

수업 시간에는 빠른 시연을 위해 기본 토크나이저를 그대로 씁니다.

## 3. CrossEncoder Re-rank — 2 단계 검색

**비유:** "서류 전형 후 면접"
1. **1단계 (서류 전형, 빠름):** BM25 + 벡터로 후보 5~10 개 추출.
2. **2단계 (면접, 정밀):** CrossEncoder 가 (질문, 문서) 쌍을 **동시에** 보고 관련성 점수를 매긴 뒤 상위 K 개만 남김.

CrossEncoder 는 bi-encoder 처럼 질문·문서를 따로 임베딩하는 게 아니라 **둘을 같이 입력** 해서 토큰 단위 어텐션으로 관련성을 계산하므로 정확도가 훨씬 높습니다 (대신 느림 → 후보 수가 적을 때만 효율적).

### 모델 선택
- `BAAI/bge-reranker-base` — 한국어 포함 다국어, 약 400MB.
- `cross-encoder/ms-marco-MiniLM-L-6-v2` — 영어 중심이지만 한국어도 준수, **~80MB 로 가볍고 빠름** → 본 노트북 기본 선택.

In [ ]:
# CrossEncoder 모델 로드 — 환경 문제로 실패할 수 있어 try/except 로 폴백.
# 처음 실행 시 모델 가중치(~80MB) 가 자동 다운로드되므로 시간이 걸립니다.
RERANK_AVAILABLE = False
try:
    from sentence_transformers import CrossEncoder
    # CrossEncoder("모델이름") 한 줄로 모델 로드 — 내부적으로 PyTorch 가중치를 load.
    # ms-marco-MiniLM-L-6-v2 는 가벼우면서 한국어도 어느 정도 처리 가능한 학습용 모델.
    reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    RERANK_AVAILABLE = True
    print("CrossEncoder loaded: cross-encoder/ms-marco-MiniLM-L-6-v2")
except Exception as e:
    # 다운로드 실패·메모리 부족 등 → 이후 셀에서 reranker is None 분기로 자동 우회.
    print(f"[WARN] CrossEncoder 로드 실패 → Re-rank 단계를 건너뜁니다.\n  사유: {e}")
    reranker = None

In [ ]:
# Re-rank 함수 — (질문, 문서) 쌍을 CrossEncoder 에 입력해 관련성 점수를 매긴다.
def rerank(query: str, documents, top_k: int = 3):
    """CrossEncoder 로 (query, doc) 쌍을 점수화하여 상위 K 개 반환.
    reranker 가 없으면 원래 순서 그대로 앞 K 개만 돌려줍니다."""
    # 폴백: reranker 가 None 이면 점수 0.0 으로 앞 K 개만 그대로 반환.
    if reranker is None or not documents:
        return [(doc, 0.0) for doc in documents[:top_k]]

    # CrossEncoder 는 (질문, 문서) 튜플 리스트를 받아 한 번에 점수 배열을 돌려줍니다.
    pairs = [(query, doc.page_content) for doc in documents]
    scores = reranker.predict(pairs)
    # zip 으로 (문서, 점수) 쌍을 만든 뒤 점수 내림차순 정렬.
    scored = list(zip(documents, scores))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]


# 실제 시연 — 1차 후보 5개를 만든 뒤 Re-rank 로 정밀 정렬.
question = "야간에 어떤 과에서 진료를 받을 수 있나요?"
candidates = ensemble_retriever.invoke(question)[:5]

print(f"❓ {question}\n")
print(f"🔍 1단계 — 하이브리드 후보 ({len(candidates)}개):")
for doc in candidates:
    print(f"  - {doc.page_content[:60]}...")

reranked = rerank(question, candidates, top_k=3)
print(f"\n🏆 2단계 — Re-rank 후 (상위 {len(reranked)}개):")
for doc, score in reranked:
    # 점수의 절대값 자체보다 "후보 간 상대적 순위 변화" 가 더 중요.
    print(f"  [{score:.4f}] {doc.page_content[:60]}...")

## 4. 하이브리드 + Re-rank RAG 체인

검색 파이프라인을 LCEL 체인에 꽂아 **end-to-end 답변 생성** 까지 연결합니다.

```
질문 → ensemble_retriever → Re-rank → 상위 3개 → LLM → 답변
```

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


def hybrid_rerank_retriever(query: str) -> str:
    """하이브리드 검색 → Re-rank → 상위 3개 문서를 컨텍스트 문자열로."""
    candidates = ensemble_retriever.invoke(query)
    if not candidates:
        return "(검색 결과 없음)"
    reranked = rerank(query, candidates, top_k=3)
    return "\n\n".join(doc.page_content for doc, _ in reranked)


rag_prompt = ChatPromptTemplate.from_template(
    "다음 컨텍스트를 바탕으로 질문에 한국어로 답변하세요.\n"
    "컨텍스트에 없는 내용은 \"해당 정보가 없습니다\" 라고 답변하세요.\n\n"
    "## 컨텍스트\n{context}\n\n"
    "## 질문\n{question}\n\n"
    "## 답변"
)

final_rag_chain = (
    {"context": hybrid_rerank_retriever, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

for q in [
    "야간에 심장 문제가 생기면 어떻게 하나요?",
    "관절 수술 전문의는 누구인가요?",
    "입원 1인실 비용과 식대는?",
]:
    print(f"\n❓ {q}\n💬 {final_rag_chain.invoke(q)}")
    print("-" * 40)

## 5. 파이프라인 단계별 비교

같은 질문에 네 가지 방식을 돌려 **체감 품질** 을 비교합니다:

1. 벡터만 → 2. BM25 만 → 3. 하이브리드 → 4. 하이브리드 + Re-rank

단계가 늘수록 정밀도는 올라가지만 **지연/비용**도 증가합니다.

In [ ]:
question = "야간에 심장 문제가 생기면?"

vec_only = vector_retriever.invoke(question)
print("🔵 벡터만:")
for doc in vec_only:
    print(f"  - {doc.page_content[:55]}...")

bm25_only = bm25_retriever.invoke(question)
print("\n🟢 BM25만:")
for doc in bm25_only:
    print(f"  - {doc.page_content[:55]}...")

hybrid = ensemble_retriever.invoke(question)
print("\n🟡 하이브리드:")
for doc in hybrid:
    print(f"  - {doc.page_content[:55]}...")

reranked = rerank(question, hybrid, top_k=3)
print("\n🏆 하이브리드 + Re-rank:")
for doc, score in reranked:
    print(f"  [{score:.4f}] {doc.page_content[:55]}...")

## 6. Weight 실험 — 최적 BM25/Vector 비율 찾기

`weights=[bm25_w, vec_w]` 비율을 0.0~1.0 까지 바꿔 Top-1 결과가 어떻게 변하는지 관찰합니다.

**경향:**
- **BM25 비중 ↑** : 고유명사·코드·정확 문자열 매칭에 유리 (의료·법률·제품 ID).
- **Vector 비중 ↑** : 유의어·개념 검색에 유리 (상담·FAQ).

In [ ]:
import pandas as pd

question = "정형외과 척추 전문의"  # 고유명사("척추") + 의미("정형외과")

print(f"❓ 질문: {question}\n")
rows = []
for bm25_w in [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]:
    vec_w = 1.0 - bm25_w
    ens = EnsembleRetriever(
        retrievers=[bm25_retriever, vector_retriever],
        weights=[bm25_w, vec_w],
    )
    results = ens.invoke(question)
    top_doc = results[0].page_content[:55] if results else "(없음)"
    rows.append({"bm25_w": bm25_w, "vec_w": round(vec_w, 2), "top_1": top_doc})

pd.DataFrame(rows)

## 7. (심화) Parent-Child Chunking — 작게 찾고 크게 답하기

**아이디어:** 검색은 **작은 청크** (정밀도 ↑) 로 하되, LLM 에 전달하는 컨텍스트는 그 청크가 속한 **큰 청크** (문맥 충분) 를 사용.

- **Child chunk (검색용):** ~150 토큰. 질문과 의미적으로 정확히 일치하는 문장만 잡기 쉬움.
- **Parent chunk (답변용):** ~1000 토큰. Child 가 속한 전체 단락을 LLM 에게 넘겨 맥락 유지.

LangChain `ParentDocumentRetriever` 가 이 패턴을 그대로 제공합니다.

In [ ]:
# Parent-Child Chunking — "작은 청크로 정밀 검색 + 큰 청크로 풍부한 컨텍스트" 트릭.
# 검색 정확도(작아야 좋음) 와 답변 풍부도(커야 좋음) 의 트레이드오프를 동시에 잡습니다.
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 데모용 긴 문서 — 한 문단에 여러 주제(의사, 진료시간, 장비)가 섞여 있다.
# Naive RAG 는 이런 큰 청크를 통째로 넘겨 LLM 의 답이 산만해집니다.
long_documents = [
    Document(
        page_content=(
            "본원 심장내과 개요. 심장내과는 관상동맥질환·부정맥·심부전을 주로 진료합니다. "
            "김철수 전문의는 15년 경력으로 관상동맥 중재술 1,200례를 보유하며 매주 수요일 야간 당직을 섭니다. "
            "외래는 평일 09:00-17:30, 심장 응급환자는 야간에도 당직의가 즉시 호출됩니다. "
            "진단 장비로는 64채널 CT, 심장 MRI, 관상동맥 조영실 2기를 운영합니다."
        ),
        metadata={"dept": "심장내과"},
    ),
    Document(
        page_content=(
            "정형외과는 척추·관절·외상을 삼분 운영합니다. 윤성호 전문의는 척추 디스크와 협착증 수술이 주력으로 "
            "내시경적 감압술을 주로 시행합니다. 한지은 전문의는 무릎·어깨 관절경 수술이 주력입니다. "
            "외상환자는 응급실을 경유합니다. 외래 예약은 홈페이지 또는 전화 02-0000-0000 으로 가능합니다."
        ),
        metadata={"dept": "정형외과"},
    ),
    Document(
        page_content=(
            "입원·편의 안내. 입원실은 1인실 25만원, 2인실 15만원, 4인실 8만원이며 식대는 1식 8천원 별도입니다. "
            "보호자 주차는 1일 5천원 정액제. 외래 환자는 3시간 무료, 이후 30분당 1천원 과금입니다. "
            "편의점·카페는 지하 1층에 24시간 운영, 코인세탁기는 병동별로 1대씩 비치돼 있습니다."
        ),
        metadata={"type": "facility"},
    ),
]

# 두 단계 분할기: parent(=큰 단위, 답변용) → child(=작은 단위, 검색용).
# RecursiveCharacterTextSplitter 는 글자 수 기준이지만 우선순위 구분자(\n\n → \n → 공백)를 지능적으로 따른다.
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
child_splitter  = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)

# 자식 청크는 별도 컬렉션으로 분리 저장 — child 문서가 parent 와 섞이지 않도록.
child_vs = Chroma(
    collection_name="parent_child_children",
    embedding_function=embeddings,
    persist_directory="./retrieval_chroma",
)
# Parent 청크는 메모리 KV 스토어에 보관 — 검색은 child_vs 로 하지만 반환은 parent 를 돌려준다.
parent_store = InMemoryStore()

# ParentDocumentRetriever 가 위 두 저장소를 묶어 준다.
pc_retriever = ParentDocumentRetriever(
    vectorstore=child_vs,                  # 검색은 자식 청크 임베딩에서
    docstore=parent_store,                  # 반환은 부모 청크 본문으로
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
    search_kwargs={"k": 2},
)
# add_documents 호출 시 parent_splitter → child_splitter 가 자동 적용되어 분할 + 인덱싱.
pc_retriever.add_documents(long_documents)

# 효과 시연: 짧은 단서가 들어간 질문 → 작은 청크가 매칭 → 그 청크가 속한 큰 단락을 통째로 반환.
q = "김철수 전문의의 야간 당직 요일은?"
hits = pc_retriever.invoke(q)
print(f"❓ {q}")
print(f"🔎 Parent 문서 {len(hits)}개 반환 (각 문서는 자식 청크가 속한 긴 문단):")
for i, doc in enumerate(hits, 1):
    print(f"\n  [{i}] ({len(doc.page_content)} chars) {doc.page_content[:200]}...")

## 실습 과제

다음 1 가지 실습을 직접 작성해 보세요. (정답 코드는 의도적으로 비워 두었습니다.)

### 1. 최적 weight 찾기 실험
BM25와 벡터의 비율을 0.0 ~ 1.0까지 변경하면서 검색 결과를 비교합니다.

질문 `"정형외과 척추 전문의"` 에 대해 BM25 가중치를 `[0.0, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0]` 으로 바꿔가며 `EnsembleRetriever` 의 Top-1 결과가 어떻게 달라지는지 표로 출력하세요.

_힌트: 반복문 안에서 매번 새 `EnsembleRetriever(retrievers=[bm25_retriever, vector_retriever], weights=[bm25_w, 1.0 - bm25_w])` 를 만들고, `invoke(question)[0].page_content[:50]` 로 Top-1 미리보기를 뽑아 한 줄씩 정렬해 출력합니다._

**결과 기록표:**

| BM25 | Vector | Top 1 결과 | 정확한가? |
|---|---|---|---|
| 0.0 | 1.0 | (벡터만) | |
| 0.3 | 0.7 | | |
| 0.4 | 0.6 | | |
| 0.5 | 0.5 | | |
| 0.7 | 0.3 | | |
| 1.0 | 0.0 | (BM25만) | |

**팁**: 고유명사가 많은 도메인(의료, 법률)은 BM25 비중을 높이고, 의미 검색이 중요한 도메인(상담, FAQ)은 벡터 비중을 높이세요.


In [ ]:
# ============================================================
# 실습 과제 — 최적 weight 찾기 실험
# ============================================================

# 실습 1: BM25/벡터 weight 조합으로 EnsembleRetriever Top-1 비교
# TODO: BM25 가중치 [0.0, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0] 로 EnsembleRetriever 를 바꿔가며 invoke(question)[0].page_content[:50] 을 표로 출력하세요.
# 여기에 구현하세요.


## 다음 노트북에서는…

**`16_langgraph_concept.ipynb`** 에서 **LangGraph** 를 소개합니다 — LCEL 의 단방향 DAG 한계를 넘어 **루프·조건 분기·재시도** 가 가능한 `StateGraph` 를 익히고, 17 번 SQL 에이전트에서 핵심적으로 쓸 **ReAct 패턴** 을 미리 맛봅니다.